# Lenguajes y Autómatas II (SCD-1016)
## Sesión 38 · Enlace (Linking) y Creación de Binarios Ejecutables ELF

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué ocurre entre el archivo .s y el programa que finalmente corre en Linux?
>
> **Objetivo de la sesión:** Seguir el recorrido as, .o, ld/gcc, ELF y comprobar dos resultados observables: símbolos sin resolver y el código de salida que ve el shell.

**Entregable:** tu implementación de codigo_salida() y simbolos_sin_resolver() en artesano_sesion38_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion38.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un valor de retorno de main o una tabla de símbolos con marca :U o :T; devuelves el código de salida que verá el shell o cuántos símbolos siguen indefinidos.

Al final de la sesión tendrás: **Dos funciones: una que predice el $? de un main y otra que cuenta los símbolos que el enlazador todavía debe resolver.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 38](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion38.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion38.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion38.h"

int codigo_salida(int retorno_main);
int simbolos_sin_resolver(const char *tabla);
```

- El código de salida es retorno & 0xFF: los valores negativos y los mayores que 255 se recortan.
- En la tabla, cada símbolo es un token separado por espacios y termina en :U (indefinido) o :T (definido).
- Una tabla vacía no tiene símbolos indefinidos; un token que no termina en :U tampoco se cuenta.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion38_c.c` junto con `artesano_sesion38_c.c`
> (o `ia_sesion38_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `codigo_salida(42)` | 42 |
| 2 | `codigo_salida(256)` | 0 |
| 3 | `codigo_salida(-1)` | 255 |
| 4 | `simbolos_sin_resolver("printf:U malloc:U main:T")` | 2 |
| 5 | `simbolos_sin_resolver("")` | 0 |
| 6 | `simbolos_sin_resolver("main:T")` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 38 - Llamada a ensamblador as y enlazador gcc ===
Caso                               esperado:              obtenido:              resultado
[ 1] codigo_salida(42)                  esperado: 42           obtenido: 42           OK
[ 2] codigo_salida(256)                 esperado: 0            obtenido: 0            OK
[ 3] codigo_salida(-1)                  esperado: 255          obtenido: 255          OK
[ 4] simbolos_sin_resolver("printf:U malloc:U main:T") esperado: 2            obtenido: 2            OK
[ 5] simbolos_sin_resolver("")          esperado: 0            obtenido: 0            OK
[ 6] simbolos_sin_resolver("main:T")    esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion38.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] codigo_salida reproduce el recorte a 8 bits que hace el sistema operativo con el retorno de main.
- [ ] simbolos_sin_resolver cuenta solo los tokens terminados en :U y tolera cadenas vacías.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion38_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion38.h
/* api_sesion38.h - contrato de la Sesion 38
 * Llamada a ensamblador as y enlazador gcc
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION38_H
#define API_SESION38_H

int codigo_salida(int retorno_main);
int simbolos_sin_resolver(const char *tabla);

#endif /* API_SESION38_H */


In [ ]:
%%writefile artesano_sesion38_c.c
/* artesano_sesion38_c.c - Fase 1: El Artesano (sin IA)
 * Llamada a ensamblador as y enlazador gcc.
 *
 * Implementa lo que declara api_sesion38.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion38.h"
#include <stdio.h>
#include <string.h>

int codigo_salida(int retorno_main) {
    (void)retorno_main; /* TODO: recortar al byte bajo del proceso */
    return 0;
}

int simbolos_sin_resolver(const char *tabla) {
    (void)tabla; /* TODO: contar los tokens terminados en :U */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion38_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion38_c.c
/* ia_sesion38_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion38.h. No puede traer main().
 */
#include "api_sesion38.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion38_c.c
/* harness_sesion38_c.c - auditoria automatica de la Sesion 38
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion38_c.c artesano_sesion38_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion38.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 38 - Llamada a ensamblador as y enlazador gcc ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = codigo_salida(42); verificar_int("codigo_salida(42)", (long)(42), (long)(r)); }
    { int r = codigo_salida(256); verificar_int("codigo_salida(256)", (long)(0), (long)(r)); }
    { int r = codigo_salida(-1); verificar_int("codigo_salida(-1)", (long)(255), (long)(r)); }
    { int r = simbolos_sin_resolver("printf:U malloc:U main:T"); verificar_int("simbolos_sin_resolver(\"printf:U malloc:U main:T\")", (long)(2), (long)(r)); }
    { int r = simbolos_sin_resolver(""); verificar_int("simbolos_sin_resolver(\"\")", (long)(0), (long)(r)); }
    { int r = simbolos_sin_resolver("main:T"); verificar_int("simbolos_sin_resolver(\"main:T\")", (long)(0), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion38.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion38_c.c artesano_sesion38_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion38_c.c ia_sesion38_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion38.md` a `auditorias/auditoria_sesion38.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
